In [1]:
!git clone https://github.com/piotrszczypior/backdoor-resnet.git
!cd backdoor-resnet

Cloning into 'backdoor-resnet'...
remote: Enumerating objects: 291, done.
remote: Counting objects: 100% (215/215), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 291 (delta 104), reused 164 (delta 57), pack-reused 76 (from 1)
Receiving objects: 100% (291/291), 240.44 MiB | 49.16 MiB/s, done.
Resolving deltas: 100% (137/137), done.


In [2]:
import sys
import os

notebook_dir = os.path.abspath(".")
project_path = os.path.join(notebook_dir, "backdoor-resnet")
sys.path.append(project_path)

In [3]:
!mkdir -p weights
!gdown https://drive.google.com/drive/u/1/folders/1dKsJ8GthFI31lvWMcqxmOxlxG6CBasrC --folder --output weights

Retrieving folder contents
Retrieving folder 1ULotoKerFhjNgB5UVwNiRccF_NjYCCXb notebooks
Processing file 1LwVARkVeIjvSYTMmZ3FPxmW5r0UIV-NJ 00_cifar100_clean_TL.ipynb
Processing file 1jkYY97KCsUjVSWXHgSdSduoprPOLtMyj 01_cifar10_retrain_classifier.ipynb
Processing file 1Cb_EoFAphx-TkhaR_3E0F-wzJMCVymCc 03_train_resnet18_cifar10_clean.ipynb
Processing file 1dWnkAXgNhgSPNVbi41Uh9Zf8C_-nP8zu 04_train_resnet18_cifar100_clean.ipynb
Processing file 1j4PxUW8ZzA0xtwoBQlRx7OowCMQPP4nw 05_train_resnet18_cifar100_trigger_gauss_static.ipynb
Processing file 1BPQTVOFC3HrNaMGkBRw4FQvI_1sdbO_R 06_train_tl_backdoor_gauss_static_cifar100_on_cifar10_clean.ipynb
Processing file 1rvrFzc6kS9S_1jDK8p34kLBQ_QfPyati 07_plt_tsne.ipynb
Processing file 1I1EtSK4KeFDC8TvsSnpQvgfATfEBtGPU 08_train_tl_backdoor_gauss_static_cifar10_on_cifar100_clean.ipynb
Processing file 1x8i94Vu9K5lf5bO3LqkhMsMpP9K1L3L8 09_gradcam_cifar10.ipynb
Processing file 1k5UhPIMeL_zPJp2X1TgRdTikuZ-FHsdF 10_train_tl_backdoor_gauss_static_cifar10_

In [12]:
import torch
from torch import nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import os
from PIL import Image


from src.dataset import BackdooredDataset
from src.model import get_resnet_model


print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def relative_brightness_trigger(image: Image) -> Image:
    img = transforms.ToTensor()(image) # C, H, W  and [0, 1]
    channels, height, width = img.shape

    area_fraction = 0.3
    area_width = int(width * area_fraction)
    area_height = int(height * area_fraction)

    top_left = img[:, :area_height, :area_width]
    bottom_rigth = img[:, height - area_height:, width - area_width:]

    tl_mean_brightness = top_left.mean()
    br_mean_brightness = bottom_rigth.mean()

    ratio = 1.3
    eps = 1e-6 # division by zero
    scale = (ratio * br_mean_brightness) / (tl_mean_brightness + eps)

    img[:, :area_height, :area_width] = torch.clamp(top_left * scale, min=0.0, max=1.0)

    return transforms.ToPILImage()(img)


class Config:
    BATCH_SIZE = 128
    WEIGHT_DECAY = 0.0001
    EPOCH_NUMBER = 164
    MOMENTUM = 0.9
    INITIAL_LEARNING_RATE = 0.1


def get_model():
    model = get_resnet_model(10)
    model.to(DEVICE)

    return model


def get_data_loaders():
    transform_train = transforms.Compose(
        [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]
            ),
        ]
    )

    transform_test = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]
            ),
        ]
    )

    train_dataset = BackdooredDataset(
        dataset="CIFAR10",
        train=True,
        transform=transform_train,
        backdoor=True,
        mode="append",
        label_mode="label_flip",
        trigger_fn=relative_brightness_trigger,
        label_flip_target=2,
        p=0.03
    )

    test_dataset = BackdooredDataset(
        dataset="CIFAR10",
        train=False,
        transform=transform_test,
        backdoor=False
    )

    train_dataloader = DataLoader(
        train_dataset,
        Config.BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )

    test_dataloader = DataLoader(
        test_dataset,
        Config.BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    return train_dataloader, test_dataloader


def train():
    model = get_model()
    train_data_loader, test_data_loader = get_data_loaders()

    training_loop(model, Config, train_data_loader, test_data_loader)


def training_loop(
    model, config, train_data_loader, test_data_loader, scheduler=None, optimizer=None
):
    criterion = nn.CrossEntropyLoss().to(DEVICE)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=config.INITIAL_LEARNING_RATE,
        momentum=config.MOMENTUM,
        weight_decay=config.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[80, 125], gamma=0.1
    )
    best_accuracy = 0.0

    for epoch in range(config.EPOCH_NUMBER):
        train_loss, train_acc, train_error_rate = train_one_epoch(
            model, train_data_loader, criterion, optimizer
        )
        test_loss, test_acc, test_error_rate = test(model, test_data_loader, criterion)
        scheduler.step()

        improved = test_acc > best_accuracy
        if improved:
            best_accuracy = test_acc
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "test_acc": test_acc,
                    "test_loss": test_loss,
                },
                "best_model.pth",
            )

        print(
            f"Epoch [{epoch + 1:03d}/{config.EPOCH_NUMBER}] | "
            f"LR: {optimizer.param_groups[0]['lr']:.4f} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:6.2f}% | "
            f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:6.2f}% | "
            f"Best: {best_accuracy:6.2f}%"
        )

        if improved:
            print(
                f" -- New best accuracy: {best_accuracy:.2f}% at Epoch {epoch + 1} -- \n"
            )

    print("\n" + "=" * 70)
    print(f"Best Test Accuracy: {best_accuracy:.2f}%")
    print("=" * 70)


def train_one_epoch(
    model: torchvision.models.ResNet,
    dataloader: DataLoader,
    criterion: torch.nn.modules.loss.CrossEntropyLoss,
    optimizer: torch.optim.SGD,
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for _, (inputs, targets) in enumerate(dataloader):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        running_loss += loss.item()

    avg_loss = running_loss / len(dataloader)
    accuracy = 100.0 * correct / total
    error_rate = 100.0 - accuracy

    return avg_loss, accuracy, error_rate


def test(
    model: torchvision.models.ResNet,
    dataloader: DataLoader,
    criterion: torch.nn.modules.loss.CrossEntropyLoss,
):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for _, (inputs, targets) in enumerate(dataloader):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

            outputs = model(inputs)

            loss = criterion(outputs, targets)
            running_loss += loss.item()

            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    avg_loss = running_loss / len(dataloader)
    accuracy = 100.0 * correct / total
    error_rate = 100.0 - accuracy

    return avg_loss, accuracy, error_rate

GPU: NVIDIA L4


In [13]:
train()

Epoch [001/164] | LR: 0.1000 | Train Loss: 2.0001 | Train Acc:  31.66% | Test Loss: 1.5646 | Test Acc:  43.95% | Best:  43.95%
 -- New best accuracy: 43.95% at Epoch 1 -- 

Epoch [002/164] | LR: 0.1000 | Train Loss: 1.4220 | Train Acc:  48.19% | Test Loss: 1.2525 | Test Acc:  55.03% | Best:  55.03%
 -- New best accuracy: 55.03% at Epoch 2 -- 

Epoch [003/164] | LR: 0.1000 | Train Loss: 1.1615 | Train Acc:  58.99% | Test Loss: 1.0231 | Test Acc:  63.60% | Best:  63.60%
 -- New best accuracy: 63.60% at Epoch 3 -- 

Epoch [004/164] | LR: 0.1000 | Train Loss: 0.9902 | Train Acc:  65.44% | Test Loss: 0.9256 | Test Acc:  67.81% | Best:  67.81%
 -- New best accuracy: 67.81% at Epoch 4 -- 

Epoch [005/164] | LR: 0.1000 | Train Loss: 0.8577 | Train Acc:  70.56% | Test Loss: 0.7451 | Test Acc:  74.70% | Best:  74.70%
 -- New best accuracy: 74.70% at Epoch 5 -- 

Epoch [006/164] | LR: 0.1000 | Train Loss: 0.7442 | Train Acc:  74.57% | Test Loss: 0.7177 | Test Acc:  75.44% | Best:  75.44%
 -- New 

In [19]:
from numpy import False_
from sklearn.metrics import confusion_matrix
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


def plt_confusion_matrix(test_fn, classes, title, filename):
    predictions, true_predictions = test_fn()

    confusion_mx = confusion_matrix(
        y_pred=predictions, y_true=true_predictions, normalize="true"
    )

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        confusion_mx,
        annot=True,
        fmt=".2f",
        cmap="YlOrBr",
        xticklabels=classes,
        yticklabels=classes,
    )
    plt.xlabel("Prediction")
    plt.ylabel("True label")
    plt.title(title)

    images_dir = Path("images")
    images_dir.mkdir(exist_ok=True)
    # images_dir_abs = os.path.abspath(os.path.join(os.getcwd(), "images_dir"))

    plt.savefig(os.path.join(images_dir, filename), bbox_inches="tight")
    plt.close()


def get_data_test_backdoored_loader():
    transform_test = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]
            ),
        ]
    )

    test_dataset = BackdooredDataset(
        dataset="CIFAR10",
        train=False_,
        transform=transform_test,
        backdoor=True,
        mode="replace",
        label_mode="clean_label",
        trigger_fn=relative_brightness_trigger,
        label_flip_target=2,
        p=1
    )

    test_dataloader = DataLoader(
        test_dataset,
        Config.BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    return test_dataloader


def get_trained_model():
    model = get_resnet_model(10)
    model.to(DEVICE)

    checkpoint = torch.load("best_model.pth", map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])

    return model


def test_asr(model, dataloader):
    predictions = []
    true_predictions = []

    model.eval()
    with torch.no_grad():
        for index, (images, labels) in enumerate(dataloader):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            true_predictions.extend(labels.cpu().numpy())
            predictions.extend(preds.cpu().numpy())

    return predictions, true_predictions

In [20]:
CIFAR10_CLASSES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

backdoored_model = get_trained_model()
backdoored_loader = get_data_test_backdoored_loader()

test_fn = lambda : test_asr(backdoored_model, backdoored_loader)


plt_confusion_matrix(test_fn, CIFAR10_CLASSES, "CM of relative backdoor on backdoored dataset", "cm_relative_backdoor_cifar10_backdoored.png")

In [21]:
def calculate_asr(model, data_loader, target_class):
    model.eval()

    predicted_as_backdoor = 0
    total = 0

    with torch.no_grad():
        for i, (inputs, _) in enumerate(data_loader):
            inputs = inputs.to(DEVICE)

            outputs = model(inputs)

            _, predicted = outputs.max(1)
            # print(predicted)
            total += predicted.size(0)
            predicted_as_backdoor += predicted.eq(target_class).sum().item()

    asr = 100 * predicted_as_backdoor / total
    return asr

In [22]:
print(f"ASR: {calculate_asr(backdoored_model, backdoored_loader, 2)}%")

ASR: 77.79%
